# Artificial Neural Networks: Fundamentals

This section introduces the foundational concepts of artificial neural networks (ANNs), also known as feedforward neural networks. Understanding these basics is essential before exploring more specialized architectures like Convolutional Neural Networks (CNNs) used for magnetic field prediction.

## Machine Learning Context

**Machine learning** enables computers to learn patterns from data without explicit programming. In **supervised learning for regression** - the paradigm used throughout this work - we train models to map inputs to continuous-valued outputs using labeled training examples. For electromagnetic field prediction, this means learning the relationship between motor geometry parameters (inputs) and magnetic flux density distributions (outputs) from thousands of FEA simulations.

**Deep learning** extends this approach using neural networks with multiple layers, enabling **hierarchical feature learning** at multiple scales:
- **Early layers**: Detect local patterns (material boundaries, flux gradients in EM)
- **Middle layers**: Combine features (saturation zones, airgap fields in EM) 
- **Late layers**: Integrate global context (complete field patterns in EM)

This hierarchical learning is why deep CNNs outperform shallow networks for spatial field prediction, as demonstrated in Sections 4b-4d.

## Neural Network Architecture

Neural Networks consist of collections of neurons that are connected in an acyclic graph. In other words, the outputs of neurons in a fully connected layer can become inputs to other neurons through pairwise connections. Cycles are not allowed since that could result in an infinite loop during the forward pass of a network. Additionally, any two neurons in a single hidden layer do not share any connections.
```{figure} ../_static/figures/CNN_NN.png
---
name: fig-ann-architecture
width: 65%
---
Artificial/Feedforward Neural Network with 2 hidden layers.
```

The architecture shown above depicts an ANN with two hidden layers. Each layer performs specific computations on the input data, progressively transforming it toward the desired output.

## Neural Networks as Function Approximators for Electromagnetic Analysis

### Motivation and Context

In computational electromagnetics, most practical problems can be viewed as mappings between **physical inputs**—geometry, material properties, and excitations—and **electromagnetic outputs** such as field distributions, forces, torque, and losses. Traditional numerical solvers evaluate this mapping by repeatedly solving Maxwell’s equations for each new design or operating condition.

Neural networks offer an alternative approach. Instead of solving the governing equations from scratch for every configuration, a neural network can be trained to **approximate the underlying input–output mapping itself**. Once trained, this surrogate model enables rapid predictions across large design spaces, making it attractive for optimization, parametric studies, and uncertainty analysis in electric machine design.

---

### Operator Viewpoint: From PDE Solvers to Learned Mappings

Let the electromagnetic operator be written as

$$
\mathcal{F} :
\{\text{geometry},\ \text{materials},\ \text{excitation}\}
\rightarrow
\{\mathbf{B}(\mathbf{x}),\ \mathbf{H}(\mathbf{x}),\ T,\ \eta\}
$$

where the outputs represent magnetic field distributions and global performance quantities such as torque and efficiency.

- In **finite element analysis**, this operator is evaluated numerically by discretizing the domain and solving a large algebraic system for each sample.
- In **machine learning**, we seek a parameterized approximation

$$
\mathcal{F}_\theta \approx \mathcal{F}
$$

where the parameters $\theta$ are learned from data generated by high-fidelity simulations or experiments.

---

### Neural Networks as Parametric Function Spaces

A neural network represents a nonlinear function of the form

$$
\mathbf{y} = \mathcal{F}_\theta(\mathbf{x}),
$$

where $\mathbf{x}$ denotes the input features (such as geometric parameters, current levels, or spatial coordinates) and $\mathbf{y}$ denotes the predicted electromagnetic quantities.

From a numerical analysis perspective, neural networks can be interpreted as **adaptive function spaces**:

- Network parameters define basis functions implicitly  
- Network depth and width control approximation capacity  
- Nonlinear activation functions enable representation of material nonlinearities and complex field interactions  

This interpretation provides a natural bridge between neural networks and classical approximation methods used in computational electromagnetics.

---

### Loss Functions as Error Functionals

Training a neural network corresponds to minimizing an error functional over a dataset:

$$
\mathcal{L}_{\text{data}} =
\frac{1}{N}
\sum_{i=1}^{N}
\left\|
\mathbf{y}_i^{\mathrm{ref}} -
\mathbf{y}_i^{\mathrm{NN}}
\right\|^2
$$

where $\mathbf{y}_i^{\mathrm{ref}}$ are reference quantities obtained from finite element simulations or measurements.

This role of the loss function is analogous to:

- Residual minimization in least-squares finite element formulations  
- Energy minimization in variational methods  

In later chapters, this formulation is extended by augmenting the data loss with **physics-based residuals**, leading to Physics-Informed Neural Networks that explicitly enforce Maxwell’s equations.

---

### Training vs. Inference: Computational Implications

A key distinction between neural networks and conventional solvers lies in the separation between **offline** and **online** computation:

| Phase | Finite Element Method | Neural Network |
|------|----------------------|----------------|
| Offline | Mesh generation and assembly | Training on simulation data |
| Online | Linear or nonlinear system solve | Single forward pass |
| Scalability | Limited by solver cost | Excellent for large parametric sweeps |

Once trained, neural networks act as **real-time surrogate solvers**, enabling rapid evaluation of electromagnetic performance across large design spaces.

---

### Scope and Limitations

This chapter introduces neural networks only to the extent required for electromagnetic field and performance prediction. Detailed treatments of optimization algorithms, statistical learning theory, and generalization bounds are beyond the scope of this text.

Readers seeking a deeper and more general foundation in deep learning are referred to standard texts such as *Deep Learning* by Goodfellow, Bengio, and Courville (2016).

The following sections build upon this foundation to introduce convolutional architectures for field prediction and physics-informed formulations that directly encode Maxwell’s equations.


In [ ]:
# Visualize activation functions and their derivatives (minimal matplotlib)

import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-5, 5, 1000)

# --- activation functions ---
def relu(z): return np.maximum(0, z)
def relu_d(z): return (z > 0).astype(float)

def sigmoid(z):
    z = np.clip(z, -50, 50)  # stable enough for plotting
    return 1 / (1 + np.exp(-z))
def sigmoid_d(z):
    s = sigmoid(z)
    return s * (1 - s)

def tanh(z): return np.tanh(z)
def tanh_d(z): return 1 - np.tanh(z) ** 2

def leaky_relu(z, alpha=0.01): return np.where(z > 0, z, alpha * z)
def leaky_relu_d(z, alpha=0.01): return np.where(z > 0, 1.0, alpha)

acts = [
    ("ReLU", relu, relu_d, (-1.5, 5.5), (-0.2, 1.2)),
    ("Sigmoid", sigmoid, sigmoid_d, (-0.2, 1.2), (-0.05, 0.30)),
    ("Tanh", tanh, tanh_d, (-1.2, 1.2), (-0.2, 1.2)),
    ("Leaky ReLU", leaky_relu, leaky_relu_d, (-1.5, 5.5), (-0.2, 1.2)),
]

def style(ax, title, ylab):
    ax.set_title(title)
    ax.set_xlabel("Input (z)")
    ax.set_ylabel(ylab)
    ax.axhline(0, lw=0.7, alpha=0.3)
    ax.axvline(0, lw=0.7, alpha=0.3)
    ax.grid(True, alpha=0.25)

fig, axes = plt.subplots(len(acts), 2, figsize=(12, 12), sharex=True)
fig.suptitle("Activation Functions and Their Derivatives", fontsize=14)

for i, (name, f, df, ylim_f, ylim_df) in enumerate(acts):
    y = f(x)
    dy = df(x)

    ax_f = axes[i, 0]
    ax_d = axes[i, 1]

    ax_f.plot(x, y)
    style(ax_f, f"{name}", "f(z)")
    ax_f.set_ylim(*ylim_f)

    ax_d.plot(x, dy, linestyle="--")
    style(ax_d, f"{name} derivative", "f'(z)")
    ax_d.set_ylim(*ylim_df)

    # Helpful reference line only on derivative plots
    ax_d.axhline(1, linestyle=":", lw=0.8, alpha=0.4)

plt.tight_layout()
plt.show()



## Activation Function Comparison (Summary)

The following summarizes commonly used activation functions in the context of **electromagnetic field, torque, and power prediction models**.

### ReLU (Rectified Linear Unit)
- Fast training with constant gradient equal to 1 for positive inputs  
- Computationally efficient and simple to implement  
- Risk of *dead neurons* when inputs remain in the negative region  

**Used as the default activation in this work.**

---

### Sigmoid
- Gradients vanish rapidly for large \(|z|\)  
- Maximum derivative limited to 0.25, leading to slow convergence  
- Rarely used in hidden layers of modern deep networks  

---

### Tanh
- Zero-centered outputs improve optimization compared to sigmoid  
- Stronger gradients near zero (maximum derivative equal to 1)  
- Still suffers from vanishing gradients for large \(|z|\)  

---

### Leaky ReLU
- Avoids dead-neuron problem by allowing a small negative slope  
- Retains computational efficiency of ReLU  
- Introduces an additional hyperparameter \(\alpha\) (typically \(\alpha = 0.01\))  

---

### Recommendation

**ReLU** is preferred as a baseline activation for deep CNNs due to its simplicity, efficiency, and stable gradient flow.  
**Leaky ReLU** is a robust alternative when dead-neuron behavior is a concern.


## From Activation Functions to Practical Neural Networks

The preceding discussion focused on **activation functions**, which define the local nonlinear behavior of individual neurons. While essential, activation functions alone do not constitute a neural network. Their true expressive power emerges only when they are **composed across layers** and trained on physically meaningful data.

At this point, we move from isolated building blocks to a **fully assembled feedforward neural network (FNN)** operating on electromagnetic simulation data generated within this work.

---

### Layers as Structured Function Composition

A feedforward neural network can be viewed as a **sequence of function compositions**, where each layer applies an affine transformation followed by a nonlinear activation:

$$
\mathbf{a}^{(l)} = f\!\left( W^{(l)} \mathbf{a}^{(l-1)} + \mathbf{b}^{(l)} \right)
$$

Each **layer** serves a distinct role:

- **Input layer**  
  Encodes physical design parameters such as geometry, material dimensions, and excitation levels.

- **Hidden layers**  
  Incrementally transform inputs into higher-level representations, capturing nonlinear interactions such as magnetic saturation, fringing effects, and coupled geometric dependencies.

- **Output layer**  
  Produces the target physical quantity of interest, such as force, torque, or field magnitude.

From an electromagnetics perspective, hidden layers can be interpreted as learning **intermediate abstractions of the electromagnetic operator**, rather than explicit physical fields.

---

### Depth and Expressive Capacity

Network **depth** plays a critical role in determining what classes of functions can be efficiently represented:

- **Shallow networks**  
  Can approximate smooth mappings but often require a large number of neurons.

- **Deeper networks**  
  Represent complex nonlinear mappings more compactly by reusing intermediate features across layers.

For electromagnetic problems, depth enables the network to disentangle:
- Local geometric effects (e.g., air-gap sensitivity),
- Material nonlinearities (e.g., saturation),
- Global force or performance trends across the design space.

This mirrors the multiscale structure inherent in Maxwell-based models, even though the network itself remains purely data-driven at this stage.

---



### Transition to a Data-Driven Electromagnetic Model

Having established the role of activations, layers, and depth, we now construct and train a **fully connected feedforward neural network** using data generated from our own finite element simulations.

The example that follows uses:
- A physically meaningful electromagnetic system (C-core electromagnet),
- Design parameters sampled across a controlled range,
- Force values computed via high-fidelity FEM analysis.

This marks the transition from theoretical foundations to **hands-on surrogate modeling**, where neural networks are treated as practical engineering tools rather than abstract learning systems.

The following sections detail:
1. Generation of the training dataset using FEM,
2. Network architecture selection,
3. Training, validation, and performance evaluation on unseen designs.


The dataset used in this example is stored in:

- `c_core_force_dataset.csv`

Each row corresponds to **one FEM simulation sample**, and contains:

- **Inputs (design/excitation parameters)**  
  - `coil_current_a` : coil current \([A]\)  
  - `armature_gap_mm` : air-gap length \([mm]\)  
  - `core_width_mm` : core width \([mm]\)  
  - `coil_turns` : number of turns \([-]\)

- **Target (label)**  
  - `force_magnitude_n` : electromagnetic force magnitude \([N]\)

In operator form, the learning problem is:

$$
\{\ I,\ g,\ w,\ N\ \} \;\longrightarrow\; F
$$

where \(I\) is coil current, \(g\) is the air gap, \(w\) is the core width, \(N\) is the number of turns, and \(F\) is the force magnitude.


The following code cells follow a clear three-step structure:

1. **Data loading and preprocessing**  
   - Read `c_core_force_dataset.csv`  
   - Train/validation/test split  
   - Feature scaling (standardization)

2. **Neural network architecture (focus of the next cell)**  
   We define a fully connected feedforward network (FNN) for regression:

   $$[4] \rightarrow [32] \rightarrow [64] \rightarrow [32] \rightarrow [1]$$

   - Input layer: 4 physical parameters  
   - Hidden layers: nonlinear feature extraction (ReLU activations)  
   - Output layer: predicted force magnitude

3. **Training and evaluation**  
   - Optimize using mean squared error loss  
   - Track training and validation curves  
   - Test-set metrics (RMSE, MAE, \(R^2\), etc.)

Finally, we generate **performance plots** to verify generalization:
- training history (loss vs epoch)  
- predicted vs true force  
- residual analysis and error distribution  
- simple sensitivity visualization

In [ ]:
# ============================================================
# Data Loading and Preprocessing
# ============================================================

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load dataset
df = pd.read_csv("c_core_force_dataset.csv")

input_features = [
    "coil_current_a",
    "armature_gap_mm",
    "core_width_mm",
    "coil_turns",
]
target_feature = "force_magnitude_n"

X = df[input_features].values
y = df[target_feature].values

print(f"Loaded {len(df)} samples")
print("Input features:", input_features)
print("Target:", target_feature)

# Train / validation / test split: 70 / 20 / 10
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.33, random_state=42
)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

# Standardization
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_norm = scaler_X.fit_transform(X_train)
X_val_norm   = scaler_X.transform(X_val)
X_test_norm  = scaler_X.transform(X_test)

y_train_norm = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
y_val_norm   = scaler_y.transform(y_val.reshape(-1, 1)).flatten()
y_test_norm  = scaler_y.transform(y_test.reshape(-1, 1)).flatten()

print("Features standardized (zero mean, unit variance)")


### Dataset Overview and Preprocessing Summary

The dataset consists of **500 finite element simulation samples**, each corresponding to a unique C-core electromagnet configuration. Four physically meaningful input parameters—coil current, air-gap length, core width, and number of turns—are used to predict the resulting electromagnetic force.

The data are partitioned into:
- **350 training samples (70%)**
- **100 validation samples (20%)**
- **50 test samples (10%)**

All input features and the target force are standardized to zero mean and unit variance prior to training. This normalization ensures balanced gradient magnitudes across parameters with different physical units and improves numerical stability during optimization.

With the data prepared and scaled, we now define the neural network architecture used to learn the force–response mapping.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Convert to tensors
X_train_tensor = torch.tensor(X_train_norm, dtype=torch.float32, device=device)
y_train_tensor = torch.tensor(y_train_norm, dtype=torch.float32, device=device).unsqueeze(1)

X_val_tensor   = torch.tensor(X_val_norm, dtype=torch.float32, device=device)
y_val_tensor   = torch.tensor(y_val_norm, dtype=torch.float32, device=device).unsqueeze(1)

X_test_tensor  = torch.tensor(X_test_norm, dtype=torch.float32, device=device)
y_test_tensor  = torch.tensor(y_test_norm, dtype=torch.float32, device=device).unsqueeze(1)

class CCoreForceANN(nn.Module):
    """
    Feedforward ANN for C-core force regression.
    Architecture: [4] -> [32] -> [64] -> [32] -> [1]
    """
    def __init__(self, input_size=4, hidden_sizes=(32, 64, 32), output_size=1):
        super().__init__()
        layers = []
        prev = input_size
        for h in hidden_sizes:
            layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        layers += [nn.Linear(prev, output_size)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model = CCoreForceANN(input_size=X_train_tensor.shape[1]).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

total_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"Total parameters: {total_params:,}")


### Neural Network Architecture Summary

The feedforward neural network defined above implements a fully connected regression model with the following structure:

$$
[4] \rightarrow [32] \rightarrow [64] \rightarrow [32] \rightarrow [1]
$$

The **four input neurons** correspond directly to the physical design parameters of the C-core electromagnet. Three hidden layers with ReLU activation functions progressively transform these inputs into higher-level nonlinear representations, while the **single output neuron** predicts the electromagnetic force magnitude.

The model contains **4,385 trainable parameters**, providing sufficient expressive capacity to capture nonlinear electromagnetic effects such as air-gap sensitivity and saturation, while remaining compact enough to mitigate overfitting on a moderate-sized dataset.

With the network architecture defined, we now proceed to train the model and evaluate its ability to generalize to unseen designs.


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# DataLoader
batch_size = 32
train_loader = DataLoader(
    TensorDataset(X_train_tensor, y_train_tensor),
    batch_size=batch_size,
    shuffle=True
)

num_epochs = 200
train_losses, val_losses = [], []

for epoch in range(1, num_epochs + 1):
    # --- train ---
    model.train()
    running = 0.0
    for xb, yb in train_loader:
        pred = model(xb)
        loss = criterion(pred, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running += loss.item() * xb.size(0)

    train_loss = running / len(train_loader.dataset)
    train_losses.append(train_loss)

    # --- validate ---
    model.eval()
    with torch.no_grad():
        val_pred = model(X_val_tensor)
        val_loss = criterion(val_pred, y_val_tensor).item()
    val_losses.append(val_loss)

    # simple progress print
    if epoch == 1 or epoch % 25 == 0 or epoch == num_epochs:
        print(f"Epoch {epoch:3d}/{num_epochs} | train {train_loss:.6f} | val {val_loss:.6f}")

# Save model weights
torch.save(model.state_dict(), "c_core_force_model.pth")
print("Saved: c_core_force_model.pth")


### Training Behavior and Convergence

The training history shows rapid reduction in loss during the early epochs, followed by a gradual convergence as the network refines its internal representation of the force–response relationship. The training loss decreases by nearly three orders of magnitude over 200 epochs, indicating effective optimization of the network parameters.

The validation loss follows a similar downward trend and remains close to the training loss throughout training, suggesting **good generalization** and no significant overfitting. Minor fluctuations in the final epochs are expected due to the finite size of the validation set.

The trained model parameters are saved to disk for subsequent evaluation and post-processing.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Predict on test (normalized -> denormalize)
model.eval()
with torch.no_grad():
    y_test_pred_norm = model(X_test_tensor).detach().cpu().numpy().flatten()

y_test_pred = scaler_y.inverse_transform(y_test_pred_norm.reshape(-1, 1)).flatten()

# Metrics (original units)
mse  = mean_squared_error(y_test, y_test_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_test_pred)
r2   = r2_score(y_test, y_test_pred)
mape = np.mean(np.abs((y_test - y_test_pred) / np.maximum(np.abs(y_test), 1e-12))) * 100

residuals = y_test_pred - y_test

print("Test set performance")
print(f"  RMSE: {rmse:.4f} N")
print(f"  MAE : {mae:.4f} N")
print(f"  R^2 : {r2:.4f}")
print(f"  MAPE: {mape:.2f}%")

# --- Plots (minimal Matplotlib) ---
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
ax.plot(train_losses, label="train")
ax.plot(val_losses, label="val")
ax.set_title("Training history")
ax.set_xlabel("epoch")
ax.set_ylabel("MSE loss (normalized)")
ax.grid(True, alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.scatter(y_test, y_test_pred, alpha=0.7)
lo = min(np.min(y_test), np.min(y_test_pred))
hi = max(np.max(y_test), np.max(y_test_pred))
ax.plot([lo, hi], [lo, hi], linestyle="--")
ax.set_title(f"Predicted vs true (R²={r2:.3f})")
ax.set_xlabel("true force [N]")
ax.set_ylabel("predicted force [N]")
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(6, 4))
ax.scatter(y_test, residuals, alpha=0.7)
ax.axhline(0, linestyle="--")
ax.set_title("Residuals vs true force")
ax.set_xlabel("true force [N]")
ax.set_ylabel("residual [N]")
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


### Test Set Performance

The trained neural network is evaluated on a held-out test set consisting of previously unseen C-core configurations. The model achieves:

- **RMSE = 0.36 N**, indicating low absolute prediction error relative to the force range  
- **MAE = 0.15 N**, confirming accurate average performance across test samples  
- **\(R^2 = 0.98\)**, demonstrating excellent agreement between predicted and simulated forces  

The high coefficient of determination indicates that the network captures the dominant nonlinear dependencies between geometry, excitation, and force.

The reported **MAPE value is not physically meaningful in this context**, as several test samples correspond to small force magnitudes. When true values approach zero, percentage-based error metrics become ill-conditioned and can grow arbitrarily large despite small absolute errors. For electromagnetic force prediction, absolute metrics such as RMSE and MAE provide a more reliable assessment of model accuracy.

Overall, the results confirm that the trained network generalizes well and serves as an accurate surrogate for finite element force computation within the sampled design space.


## Summary and Outlook: From Scalar Surrogates to Field-Based Models

In this notebook, we developed a complete **data-driven surrogate modeling workflow** for a C-core electromagnet, starting from finite element simulations and culminating in a trained feedforward neural network capable of predicting electromagnetic force from a small set of design parameters.

Specifically, we demonstrated:
- Generation of a physics-based dataset using FEM,
- Construction of a fully connected neural network for regression,
- Training, validation, and evaluation of the model on unseen designs,
- Interpretation of performance using physically meaningful error metrics.

This example highlights how **feedforward neural networks** can serve as efficient surrogates for repeated electromagnetic simulations when the quantity of interest is a **scalar performance metric** such as force, torque, or loss.

---

## Limitations of Fully Connected Networks for Spatial Electromagnetic Data

Despite their effectiveness for low-dimensional regression problems, fully connected neural networks are fundamentally limited when applied to **spatially distributed data**, such as magnetic flux density fields or vector potential maps.

Key limitations include:

1. **Parameter Explosion**  
   Treating a two-dimensional field as a flat vector leads to an impractical number of parameters. For example, a \(256 \times 256\) field map connected to just 100 hidden neurons would require over 6 million weights in the first layer alone.

2. **Lack of Spatial Awareness**  
   Fully connected layers do not exploit the locality of electromagnetic phenomena. Features such as air-gap flux patterns or saturation regions must be learned independently at every spatial location.

3. **Loss of Geometric Structure**  
   Flattening spatial data destroys neighborhood relationships, obscuring the physical structure inherent in field solutions governed by Maxwell’s equations.

These limitations make fully connected architectures poorly suited for learning **field distributions**, even though they perform well for global scalar outputs.

---

## Motivation for Convolutional Neural Networks

To model spatial electromagnetic fields efficiently and accurately, the network architecture must respect the **geometry and locality** of the data. Convolutional Neural Networks (CNNs) address these challenges by:

- Exploiting **local connectivity**,  
- Sharing parameters across spatial locations via convolution filters,  
- Preserving spatial structure throughout the network depth.

These properties align naturally with the spatial structure of electromagnetic field solutions and enable scalable learning of high-resolution field maps.

The next section introduces **convolutional neural networks** in detail and demonstrates how they are applied to magnetic field prediction in electric machines.
